In [1]:
import pandas as pd
import json
import re
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import numpy as np

# Load data
df = pd.read_csv('../data/USvideos.csv', encoding='utf-8', on_bad_lines='skip')

with open('../data/US_category_id.json') as f:
    categories_json = json.load(f)

# Map categories
id_to_category = {int(cat['id']): cat['snippet']['title'] 
                  for cat in categories_json['items']}
df['category_name'] = df['category_id'].map(id_to_category)

print(f"✅ Data loaded: {len(df):,} videos")

✅ Data loaded: 40,949 videos


In [2]:
def clean_text(text):
    """
    Clean and normalize text for ML processing
    """
    if pd.isna(text):
        return ""
    
    # Convert to string and lowercase
    text = str(text).lower()
    
    # Remove URLs
    text = re.sub(r'http\S+|www\S+', '', text)
    
    # Remove HTML tags
    text = re.sub(r'<.*?>', '', text)
    
    # Remove special characters (keep only letters and spaces)
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)
    
    # Remove extra whitespace
    text = ' '.join(text.split())
    
    return text

# Test function
test_text = "Check out my NEW video! 🔥 http://example.com #trending"
print(f"Original: {test_text}")
print(f"Cleaned:  {clean_text(test_text)}")

Original: Check out my NEW video! 🔥 http://example.com #trending
Cleaned:  check out my new video trending


In [3]:
print("🧹 Cleaning text data...")

# Clean titles
df['title_clean'] = df['title'].apply(clean_text)

# Clean descriptions (handle missing values)
df['description'] = df['description'].fillna('')
df['description_clean'] = df['description'].apply(clean_text)

# Combine title and description
df['text'] = df['title_clean'] + ' ' + df['description_clean']

print("✅ Text cleaning complete!")
print(f"\nSample cleaned text:")
print(df['text'].iloc[0][:300] + "...")

🧹 Cleaning text data...
✅ Text cleaning complete!

Sample cleaned text:
we want to talk about our marriage shantell s channel this video in k on this this lens drone gear camera camera lens sony camera canon camera tripod thing need this for the bendy tripod lens expensive wide lens camera microphone drone cheaper but still great me on intro song by disclosure this is n...


In [4]:
print(f"Before filtering: {len(df):,} videos\n")

# Remove very short texts (less than 20 characters)
df = df[df['text'].str.len() > 20]
print(f"After removing short texts: {len(df):,} videos")

# Remove duplicates based on video_id
df = df.drop_duplicates(subset=['video_id'])
print(f"After removing duplicates: {len(df):,} videos")

# Remove videos with missing category
df = df.dropna(subset=['category_name'])
print(f"After removing missing categories: {len(df):,} videos")

# Keep top 10 categories for balanced dataset
top_categories = df['category_name'].value_counts().head(10).index
df = df[df['category_name'].isin(top_categories)]
print(f"After keeping top 10 categories: {len(df):,} videos")

print(f"\n✅ Final dataset: {len(df):,} videos")
print(f"\nFinal category distribution:")
print(df['category_name'].value_counts())

Before filtering: 40,949 videos

After removing short texts: 40,851 videos
After removing duplicates: 6,332 videos
After removing missing categories: 6,332 videos
After keeping top 10 categories: 5,948 videos

✅ Final dataset: 5,948 videos

Final category distribution:
category_name
Entertainment           1617
Music                    798
Howto & Style            595
Comedy                   544
News & Politics          504
People & Blogs           495
Sports                   450
Science & Technology     379
Film & Animation         318
Education                248
Name: count, dtype: int64


In [6]:
# Create label encoder
le = LabelEncoder()
df['label'] = le.fit_transform(df['category_name'])

# Create and save label mapping
label_mapping = dict(zip(le.classes_, range(len(le.classes_))))

print("✅ Labels encoded!")
print(f"\nLabel Mapping:")
for category, label_id in sorted(label_mapping.items()):
    print(f"  {label_id}: {category}")

# Save mapping for later use
import json
with open('../models/label_mapping.json', 'w') as f:
    json.dump(label_mapping, f, indent=2)

print("\n✅ Label mapping saved to: ../models/label_mapping.json")

✅ Labels encoded!

Label Mapping:
  0: Comedy
  1: Education
  2: Entertainment
  3: Film & Animation
  4: Howto & Style
  5: Music
  6: News & Politics
  7: People & Blogs
  8: Science & Technology
  9: Sports

✅ Label mapping saved to: ../models/label_mapping.json


In [7]:
# Select columns for training
data_clean = df[['text', 'label', 'category_name']].copy()

# Split: 80% training, 20% testing
train_df, test_df = train_test_split(
    data_clean,
    test_size=0.2,
    stratify=data_clean['label'],  # Maintain category distribution
    random_state=42
)

print("✅ Data split complete!")
print(f"\nTraining set: {len(train_df):,} videos ({len(train_df)/len(data_clean)*100:.1f}%)")
print(f"Testing set:  {len(test_df):,} videos ({len(test_df)/len(data_clean)*100:.1f}%)")

print(f"\nTraining set category distribution:")
print(train_df['category_name'].value_counts())

✅ Data split complete!

Training set: 4,758 videos (80.0%)
Testing set:  1,190 videos (20.0%)

Training set category distribution:
category_name
Entertainment           1294
Music                    638
Howto & Style            476
Comedy                   435
News & Politics          403
People & Blogs           396
Sports                   360
Science & Technology     303
Film & Animation         254
Education                199
Name: count, dtype: int64


In [8]:
# Save to CSV files
train_df.to_csv('../data/train_data.csv', index=False)
test_df.to_csv('../data/test_data.csv', index=False)

print("✅ Preprocessed data saved!")
print("\nFiles created:")
print("  ✓ ../data/train_data.csv")
print("  ✓ ../data/test_data.csv")
print("  ✓ ../models/label_mapping.json")
print("\n🚀 Ready for model training!")

✅ Preprocessed data saved!

Files created:
  ✓ ../data/train_data.csv
  ✓ ../data/test_data.csv
  ✓ ../models/label_mapping.json

🚀 Ready for model training!


In [3]:
import pandas as pd
import json
import re
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

print("="*70)
print("FIXED PREPROCESSING - BETTER DATA QUALITY")
print("="*70)

# 1. Load RAW dataset (start fresh)
df = pd.read_csv('../data/USvideos.csv', encoding='utf-8', on_bad_lines='skip')

with open('../data/US_category_id.json') as f:
    categories_json = json.load(f)

# Map categories
id_to_category = {int(cat['id']): cat['snippet']['title'] 
                  for cat in categories_json['items']}
df['category_name'] = df['category_id'].map(id_to_category)

print(f"✅ Original dataset: {len(df):,} videos")

# 2. IMPROVED TEXT CLEANING (keep more words!)
def clean_text(text):
    """Clean text but keep important words"""
    if pd.isna(text):
        return ""
    
    text = str(text).lower()
    
    # Remove URLs only
    text = re.sub(r'http\S+|www\S+', '', text)
    
    # Keep letters, numbers, and spaces (more lenient!)
    text = re.sub(r'[^a-zA-Z0-9\s]', ' ', text)
    
    # Remove extra whitespace
    text = ' '.join(text.split())
    
    return text

# 3. Apply cleaning
print("\n🧹 Cleaning text...")
df['title_clean'] = df['title'].apply(clean_text)
df['description'] = df['description'].fillna('')
df['description_clean'] = df['description'].apply(clean_text)

# Combine title and description
df['text'] = df['title_clean'] + ' ' + df['description_clean']

# 4. STRICT filtering (quality over quantity)
print(f"\n📊 Filtering data...")
print(f"Before filtering: {len(df):,}")

# Remove SHORT texts (less meaningful)
df = df[df['text'].str.len() >= 50]  # At least 50 characters
print(f"After removing short texts: {len(df):,}")

# Remove duplicates
df = df.drop_duplicates(subset=['video_id'])
print(f"After removing duplicates: {len(df):,}")

# Remove missing categories
df = df.dropna(subset=['category_name'])
print(f"After removing missing categories: {len(df):,}")

# 5. BALANCE DATASET (crucial for good training!)
print(f"\n⚖️ Balancing dataset...")

# Keep top 10 categories
top_categories = df['category_name'].value_counts().head(10).index
df = df[df['category_name'].isin(top_categories)]

# Balance: take equal samples from each category (prevents bias!)
min_samples = df['category_name'].value_counts().min()
balanced_samples = min(min_samples, 3000)  # Max 3000 per category

df_balanced = df.groupby('category_name', group_keys=False).apply(
    lambda x: x.sample(n=min(len(x), balanced_samples), random_state=42)
)

print(f"✅ Balanced dataset: {len(df_balanced):,} videos")
print(f"   Samples per category: ~{balanced_samples}")

# 6. Check text quality
print(f"\n📝 Text Quality Check:")
print(f"   Average length: {df_balanced['text'].str.len().mean():.0f} chars")
print(f"   Min length: {df_balanced['text'].str.len().min()}")
print(f"   Max length: {df_balanced['text'].str.len().max()}")

# 7. Show distribution
print(f"\n📊 Final Category Distribution:")
print(df_balanced['category_name'].value_counts().sort_index())

# 8. Encode labels
le = LabelEncoder()
df_balanced['label'] = le.fit_transform(df_balanced['category_name'])

# Save label mapping
label_mapping = dict(zip(le.classes_, range(len(le.classes_))))
with open('../models/label_mapping.json', 'w') as f:
    json.dump(label_mapping, f, indent=2)

print(f"\n✅ Label mapping:")
for cat, label in sorted(label_mapping.items()):
    count = len(df_balanced[df_balanced['category_name'] == cat])
    print(f"   {label}: {cat} ({count} samples)")

# 9. Split data (stratified for balance)
train_df, test_df = train_test_split(
    df_balanced[['text', 'label', 'category_name']],
    test_size=0.2,
    stratify=df_balanced['label'],
    random_state=42
)

# 10. Save
train_df.to_csv('../data/train_data.csv', index=False)
test_df.to_csv('../data/test_data.csv', index=False)

print(f"\n" + "="*70)
print("✅ PREPROCESSING COMPLETE!")
print("="*70)
print(f"\nTraining: {len(train_df):,} samples")
print(f"Testing: {len(test_df):,} samples")

# Show sample
print(f"\n📝 Sample Training Data:")
for i in range(3):
    print(f"\n{i+1}. Category: {train_df.iloc[i]['category_name']}")
    print(f"   Text: {train_df.iloc[i]['text'][:100]}...")

print("\n🚀 Ready for training!")

FIXED PREPROCESSING - BETTER DATA QUALITY
✅ Original dataset: 40,949 videos

🧹 Cleaning text...

📊 Filtering data...
Before filtering: 40,949
After removing short texts: 40,344
After removing duplicates: 6,250
After removing missing categories: 6,250

⚖️ Balancing dataset...
✅ Balanced dataset: 2,470 videos
   Samples per category: ~247

📝 Text Quality Check:
   Average length: 697 chars
   Min length: 50
   Max length: 4748

📊 Final Category Distribution:
category_name
Comedy                  247
Education               247
Entertainment           247
Film & Animation        247
Howto & Style           247
Music                   247
News & Politics         247
People & Blogs          247
Science & Technology    247
Sports                  247
Name: count, dtype: int64

✅ Label mapping:
   0: Comedy (247 samples)
   1: Education (247 samples)
   2: Entertainment (247 samples)
   3: Film & Animation (247 samples)
   4: Howto & Style (247 samples)
   5: Music (247 samples)
   6: News & 

C:\Users\AmrHassan\AppData\Local\Temp\ipykernel_17596\2678609049.py:79: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_balanced = df.groupby('category_name', group_keys=False).apply(


In [6]:
import pandas as pd
import joblib
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
import numpy as np

print("="*70)
print("TRAINING WITH IMPROVED DATA")
print("="*70)

# 1. Load data
train_df = pd.read_csv('../data/train_data.csv')
test_df = pd.read_csv('../data/test_data.csv')

print(f"\n📊 Data loaded:")
print(f"   Training: {len(train_df):,}")
print(f"   Testing: {len(test_df):,}")

# 2. Create TF-IDF (with better parameters)
vectorizer = TfidfVectorizer(
    max_features=8000,        # More features for better learning
    min_df=3,                 # Word must appear in 3+ docs
    max_df=0.7,               # Ignore very common words
    ngram_range=(1, 3),       # Include 1,2,3-word phrases
    stop_words='english',
    sublinear_tf=True,
    use_idf=True
)

# 3. Transform
print("\n🔄 Creating features...")
X_train = vectorizer.fit_transform(train_df['text'])
X_test = vectorizer.transform(test_df['text'])

y_train = train_df['label'].values
y_test = test_df['label'].values

print(f"✅ Feature matrix: {X_train.shape}")

# 4. Train with better parameters
print("\n🔄 Training model...")
model = LogisticRegression(
    max_iter=2000,
    C=2.0,
    solver='saga',
    penalty='l2',
    # multi_class parameter removed; it will default to 'auto'
    random_state=42,
    verbose=1
)

model.fit(X_train, y_train)
print("\n✅ Training complete!")

# 5. Evaluate
y_pred_train = model.predict(X_train)
y_pred_test = model.predict(X_test)

train_acc = accuracy_score(y_train, y_pred_train)
test_acc = accuracy_score(y_test, y_pred_test)

print(f"\n📊 Performance:")
print(f"   Training Accuracy: {train_acc:.4f} ({train_acc*100:.2f}%)")
print(f"   Testing Accuracy: {test_acc:.4f} ({test_acc*100:.2f}%)")

# 6. Check confidence
probs_test = model.predict_proba(X_test)
max_probs = probs_test.max(axis=1)

print(f"\n📊 Confidence Statistics:")
print(f"   Average: {max_probs.mean():.4f} ({max_probs.mean()*100:.2f}%)")
print(f"   Median: {np.median(max_probs):.4f} ({np.median(max_probs)*100:.2f}%)")
print(f"   Min: {max_probs.min():.4f}")
print(f"   Max: {max_probs.max():.4f}")

# 7. Test with specific example
test_text = "How to build a gaming PC step by step tutorial"
test_vec = vectorizer.transform([test_text])
pred = model.predict(test_vec)[0]
prob = model.predict_proba(test_vec)[0]
confidence = prob[pred]

print(f"\n🧪 Test Prediction:")
print(f"   Text: {test_text}")
print(f"   Category: {train_df[train_df['label']==pred].iloc[0]['category_name']}")
print(f"   Confidence: {confidence:.4f} ({confidence*100:.2f}%)")
print(f"\n   All probabilities:")
for i, p in enumerate(prob):
    if p > 0.05:  # Only show > 5%
        cat = train_df[train_df['label']==i].iloc[0]['category_name']
        print(f"      {cat}: {p:.4f} ({p*100:.2f}%)")

# 8. Only save if good enough
if max_probs.mean() > 0.65 and test_acc > 0.70:
    print("\n✅ Model quality is GOOD - saving...")
    joblib.dump(model, '../models/logistic_regression_model.pkl')
    joblib.dump(model, '../models/best_model.pkl')
    joblib.dump(vectorizer, '../models/tfidf_vectorizer.pkl')
    print("✅ Saved!")
    print("\n🔄 Restart Python API now!")
else:
    print(f"\n⚠️ Model quality is LOW:")
    print(f"   Average confidence: {max_probs.mean()*100:.2f}% (need >65%)")
    print(f"   Test accuracy: {test_acc*100:.2f}% (need >70%)")
    print(f"   NOT saving - check your data!")


TRAINING WITH IMPROVED DATA

📊 Data loaded:
   Training: 1,976
   Testing: 494

🔄 Creating features...
✅ Feature matrix: (1976, 8000)

🔄 Training model...


C:\Users\AmrHassan\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(


convergence after 23 epochs took 0 seconds

✅ Training complete!

📊 Performance:
   Training Accuracy: 0.9782 (97.82%)
   Testing Accuracy: 0.7510 (75.10%)

📊 Confidence Statistics:
   Average: 0.4787 (47.87%)
   Median: 0.4372 (43.72%)
   Min: 0.1298
   Max: 0.9622

🧪 Test Prediction:
   Text: How to build a gaming PC step by step tutorial
   Category: Science & Technology
   Confidence: 0.2972 (29.72%)

   All probabilities:
      Comedy: 0.0575 (5.75%)
      Entertainment: 0.0826 (8.26%)
      Film & Animation: 0.0653 (6.53%)
      Howto & Style: 0.1581 (15.81%)
      News & Politics: 0.0886 (8.86%)
      People & Blogs: 0.1144 (11.44%)
      Science & Technology: 0.2972 (29.72%)

⚠️ Model quality is LOW:
   Average confidence: 47.87% (need >65%)
   Test accuracy: 75.10% (need >70%)
   NOT saving - check your data!


In [8]:
import pandas as pd
import json
import re
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import numpy as np

print("="*70)
print("OPTIMIZED PREPROCESSING - KEEP MORE DATA")
print("="*70)

# 1. Load dataset
df = pd.read_csv('../data/USvideos.csv', encoding='utf-8', on_bad_lines='skip')

with open('../data/US_category_id.json') as f:
    categories_json = json.load(f)

id_to_category = {int(cat['id']): cat['snippet']['title'] 
                  for cat in categories_json['items']}
df['category_name'] = df['category_id'].map(id_to_category)

print(f"\n✅ Original dataset: {len(df):,} videos")

# 2. Clean text
def clean_text(text):
    if pd.isna(text):
        return ""
    text = str(text).lower()
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    text = ' '.join(text.split())
    return text

print("\n🧹 Cleaning text...")
df['title_clean'] = df['title'].apply(clean_text)
df['description'] = df['description'].fillna('')
df['description_clean'] = df['description'].apply(clean_text)
df['text'] = df['title_clean'] + ' ' + df['description_clean']

# 3. Basic filtering
df = df[df['text'].str.len() >= 50]
df = df.dropna(subset=['category_name'])
print(f"After basic filtering: {len(df):,}")

# 4. Keep top 10 categories
top_categories = df['category_name'].value_counts().head(10).index
df = df[df['category_name'].isin(top_categories)]
print(f"After keeping top 10 categories: {len(df):,}")

# 5. SMART DEDUPLICATION: Keep duplicates but limit per title
# (Same video trending on different days = useful training data!)
print(f"\n🔍 Smart deduplication...")
print(f"Before: {len(df):,}")

# Group by title and keep max 3 entries per title
df_deduplicated = df.groupby('title', group_keys=False).apply(
    lambda x: x.head(3) if len(x) > 3 else x,
    include_groups=True
).reset_index(drop=True)

print(f"After smart dedup: {len(df_deduplicated):,}")

# 6. SAMPLE MORE DATA PER CATEGORY
print(f"\n⚖️ Sampling data per category...")
category_counts = df_deduplicated['category_name'].value_counts()
print(f"\nCategory counts before sampling:")
print(category_counts)

# Take up to 3000 samples per category (or all if less)
df_sampled = df_deduplicated.groupby('category_name', group_keys=False).apply(
    lambda x: x.sample(n=min(len(x), 3000), random_state=42),
    include_groups=True
).reset_index(drop=True)

print(f"\n✅ Final dataset: {len(df_sampled):,} videos")

# 7. Show distribution
print(f"\n📊 Final Category Distribution:")
final_dist = df_sampled['category_name'].value_counts().sort_index()
print(final_dist)

# 8. Text quality
print(f"\n📝 Text Quality:")
print(f"   Average length: {df_sampled['text'].str.len().mean():.0f} chars")
print(f"   Min length: {df_sampled['text'].str.len().min()}")
print(f"   Max length: {df_sampled['text'].str.len().max()}")

# 9. Encode labels
le = LabelEncoder()
df_sampled['label'] = le.fit_transform(df_sampled['category_name'])

label_mapping = dict(zip(le.classes_, range(len(le.classes_))))
with open('../models/label_mapping.json', 'w') as f:
    json.dump(label_mapping, f, indent=2)

print(f"\n✅ Label mapping saved")

# 10. Split data (80/20)
train_df, test_df = train_test_split(
    df_sampled[['text', 'label', 'category_name']],
    test_size=0.2,
    stratify=df_sampled['label'],
    random_state=42
)

# 11. Save
train_df.to_csv('../data/train_data.csv', index=False)
test_df.to_csv('../data/test_data.csv', index=False)

print(f"\n" + "="*70)
print("✅ PREPROCESSING COMPLETE!")
print("="*70)
print(f"\n📊 Dataset Summary:")
print(f"   Training: {len(train_df):,} samples")
print(f"   Testing: {len(test_df):,} samples")
print(f"   Total: {len(df_sampled):,} samples")

# Quality check
if len(train_df) < 8000:
    print(f"\n⚠️  WARNING: Only {len(train_df):,} training samples")
    print(f"   Recommended: 10,000+ for good model performance")
    print(f"   Your model may have lower accuracy/confidence")
elif len(train_df) < 15000:
    print(f"\n✅ OK: {len(train_df):,} training samples")
    print(f"   Should work reasonably well")
else:
    print(f"\n✅ EXCELLENT: {len(train_df):,} training samples")
    print(f"   Great for high-quality model training")

# 12. Show samples
print(f"\n📝 Sample Data:")
for i in range(3):
    print(f"\n{i+1}. Category: {train_df.iloc[i]['category_name']}")
    print(f"   Text: {train_df.iloc[i]['text'][:120]}...")

print("\n🚀 Ready for training!")

OPTIMIZED PREPROCESSING - KEEP MORE DATA

✅ Original dataset: 40,949 videos

🧹 Cleaning text...
After basic filtering: 40,344
After keeping top 10 categories: 37,756

🔍 Smart deduplication...
Before: 37,756


C:\Users\AmrHassan\AppData\Local\Temp\ipykernel_17596\2417062825.py:56: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_deduplicated = df.groupby('title', group_keys=False).apply(
C:\Users\AmrHassan\AppData\Local\Temp\ipykernel_17596\2417062825.py:70: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_sampled = df_deduplicated.groupby('category_name', group_keys=False).apply(


After smart dedup: 15,912

⚖️ Sampling data per category...

Category counts before sampling:
category_name
Entertainment           4219
Music                   2282
Howto & Style           1667
Comedy                  1410
People & Blogs          1322
News & Politics         1283
Sports                  1105
Science & Technology    1032
Film & Animation         885
Education                707
Name: count, dtype: int64

✅ Final dataset: 14,693 videos

📊 Final Category Distribution:
category_name
Comedy                  1410
Education                707
Entertainment           3000
Film & Animation         885
Howto & Style           1667
Music                   2282
News & Politics         1283
People & Blogs          1322
Science & Technology    1032
Sports                  1105
Name: count, dtype: int64

📝 Text Quality:
   Average length: 701 chars
   Min length: 50
   Max length: 4820

✅ Label mapping saved

✅ PREPROCESSING COMPLETE!

📊 Dataset Summary:
   Training: 11,754 samples


In [10]:
import pandas as pd
import json
import re
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

print("="*70)
print("OPTIMIZED PREPROCESSING - MORE DATA")
print("="*70)

# 1. Load dataset
df = pd.read_csv('../data/USvideos.csv', encoding='utf-8', on_bad_lines='skip')

with open('../data/US_category_id.json') as f:
    categories_json = json.load(f)

id_to_category = {int(cat['id']): cat['snippet']['title'] 
                  for cat in categories_json['items']}
df['category_name'] = df['category_id'].map(id_to_category)

print(f"\n✅ Original: {len(df):,} videos")

# 2. Clean text
def clean_text(text):
    if pd.isna(text):
        return ""
    text = str(text).lower()
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    text = ' '.join(text.split())
    return text

print("🧹 Cleaning...")
df['title_clean'] = df['title'].apply(clean_text)
df['description'] = df['description'].fillna('')
df['description_clean'] = df['description'].apply(clean_text)
df['text'] = df['title_clean'] + ' ' + df['description_clean']

# 3. Filter
df = df[df['text'].str.len() >= 50]
df = df.dropna(subset=['category_name'])

# 4. Top 10 categories
top_10 = df['category_name'].value_counts().head(10).index
df = df[df['category_name'].isin(top_10)]
print(f"After filtering: {len(df):,}")

# 5. CRITICAL: Smart dedup - keep MORE data
print("\n🔍 Smart deduplication...")
# Keep up to 5 trending instances per video title
df_dedup = df.groupby('title', group_keys=False).apply(
    lambda x: x.head(5),
    include_groups=True
).reset_index(drop=True)

print(f"After dedup: {len(df_dedup):,}")

# 6. Sample 3000 per category (or all if less)
print("\n📊 Sampling per category...")
df_final = df_dedup.groupby('category_name', group_keys=False).apply(
    lambda x: x.sample(n=min(len(x), 3000), random_state=42),
    include_groups=True
).reset_index(drop=True)

print(f"\n✅ Final dataset: {len(df_final):,} videos")
print(f"\nCategory distribution:")
print(df_final['category_name'].value_counts().sort_index())

# 7. Encode
le = LabelEncoder()
df_final['label'] = le.fit_transform(df_final['category_name'])

with open('../models/label_mapping.json', 'w') as f:
    json.dump(dict(zip(le.classes_, range(len(le.classes_)))), f, indent=2)

# 8. Split
train_df, test_df = train_test_split(
    df_final[['text', 'label', 'category_name']],
    test_size=0.2,
    stratify=df_final['label'],
    random_state=42
)

train_df.to_csv('../data/train_data.csv', index=False)
test_df.to_csv('../data/test_data.csv', index=False)

print(f"\n{'='*70}")
print(f"✅ DONE!")
print(f"{'='*70}")
print(f"Training: {len(train_df):,} samples")
print(f"Testing: {len(test_df):,} samples")

if len(train_df) >= 10000:
    print("\n✅ EXCELLENT dataset size!")
elif len(train_df) >= 5000:
    print("\n✅ GOOD dataset size")
else:
    print(f"\n⚠️ Dataset still small ({len(train_df):,})")
    print("   But should be better than before!")

OPTIMIZED PREPROCESSING - MORE DATA

✅ Original: 40,949 videos
🧹 Cleaning...
After filtering: 37,756

🔍 Smart deduplication...


C:\Users\AmrHassan\AppData\Local\Temp\ipykernel_17596\1274644404.py:51: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_dedup = df.groupby('title', group_keys=False).apply(
C:\Users\AmrHassan\AppData\Local\Temp\ipykernel_17596\1274644404.py:60: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_final = df_dedup.groupby('category_name', group_keys=False).apply(


After dedup: 23,749

📊 Sampling per category...

✅ Final dataset: 19,947 videos

Category distribution:
category_name
Comedy                  2112
Education               1085
Entertainment           3000
Film & Animation        1364
Howto & Style           2522
Music                   3000
News & Politics         1816
People & Blogs          1985
Science & Technology    1527
Sports                  1536
Name: count, dtype: int64

✅ DONE!
Training: 15,957 samples
Testing: 3,990 samples

✅ EXCELLENT dataset size!


In [11]:
import pandas as pd
import json
import re
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

print("="*70)
print("FINAL PREPROCESSING - 8 STRONG CATEGORIES")
print("="*70)

# 1. Load data
df = pd.read_csv('../data/USvideos.csv', encoding='utf-8', on_bad_lines='skip')

with open('../data/US_category_id.json') as f:
    categories_json = json.load(f)

id_to_category = {int(cat['id']): cat['snippet']['title'] 
                  for cat in categories_json['items']}
df['category_name'] = df['category_id'].map(id_to_category)

print(f"Original: {len(df):,}")

# 2. Clean text
def clean_text(text):
    if pd.isna(text):
        return ""
    text = str(text).lower()
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    text = ' '.join(text.split())
    return text

df['title_clean'] = df['title'].apply(clean_text)
df['description'] = df['description'].fillna('')
df['description_clean'] = df['description'].apply(clean_text)
df['text'] = df['title_clean'] + ' ' + df['description_clean']

df = df[df['text'].str.len() >= 50]
df = df.dropna(subset=['category_name'])

# 3. MERGE WEAK CATEGORIES
print("\n🔀 Merging similar/weak categories...")

category_mapping = {
    'Music': 'Music',
    'Sports': 'Sports',
    'Entertainment': 'Entertainment',
    'Comedy': 'Comedy',
    'News & Politics': 'News & Politics',
    'People & Blogs': 'People & Blogs',
    'Howto & Style': 'Howto & Style',
    'Gaming': 'Entertainment',
    'Education': 'Science & Technology',
    'Science & Technology': 'Science & Technology',
    'Film & Animation': 'Entertainment'
}

df['category_merged'] = df['category_name'].map(category_mapping)
df = df.dropna(subset=['category_merged'])

print(f"After merging: {len(df):,}")
print(f"\nMerged categories:")
print(df['category_merged'].value_counts().sort_index())

# 4. FIXED: Smart dedup without losing columns
print(f"\n🔍 Deduplicating...")
df_dedup = []
for title, group in df.groupby('title'):
    # Take up to 5 entries per title
    df_dedup.append(group.head(5))

df_dedup = pd.concat(df_dedup, ignore_index=True)

print(f"After dedup: {len(df_dedup):,}")

# 5. FIXED: Balanced sampling without losing columns
print(f"\n⚖️ Sampling per category...")
df_samples = []
for category, group in df_dedup.groupby('category_merged'):
    n_sample = min(len(group), 2500)
    sampled = group.sample(n=n_sample, random_state=42)
    df_samples.append(sampled)
    print(f"   {category}: {n_sample} samples")

df_final = pd.concat(df_samples, ignore_index=True)

print(f"\n✅ Final dataset: {len(df_final):,} videos")
print(f"\nFinal distribution:")
final_dist = df_final['category_merged'].value_counts().sort_index()
print(final_dist)

print(f"\n📊 Balance check:")
print(f"   Min: {final_dist.min()}")
print(f"   Max: {final_dist.max()}")
print(f"   Difference: {final_dist.max() - final_dist.min()}")

if final_dist.max() - final_dist.min() < 1000:
    print("   ✅ Well balanced!")
else:
    print("   ⚠️ Some imbalance remains")

# 6. Encode labels
le = LabelEncoder()
df_final['label'] = le.fit_transform(df_final['category_merged'])

label_mapping = dict(zip(le.classes_, range(len(le.classes_))))

print(f"\n📋 Final Label Mapping:")
for cat, label_id in sorted(label_mapping.items()):
    count = len(df_final[df_final['category_merged'] == cat])
    print(f"   {label_id}: {cat} ({count} samples)")

# Save label mapping
with open('../models/label_mapping.json', 'w') as f:
    json.dump(label_mapping, f, indent=2)

print("   ✅ Saved to label_mapping.json")

# 7. Prepare for splitting
df_for_split = df_final[['text', 'label', 'category_merged']].copy()
df_for_split.rename(columns={'category_merged': 'category_name'}, inplace=True)

# 8. Split into train/test
train_df, test_df = train_test_split(
    df_for_split,
    test_size=0.2,
    stratify=df_for_split['label'],
    random_state=42
)

# 9. Save to CSV
train_df.to_csv('../data/train_data.csv', index=False)
test_df.to_csv('../data/test_data.csv', index=False)

print(f"\n{'='*70}")
print("✅ PREPROCESSING COMPLETE - 8 STRONG CATEGORIES")
print("="*70)
print(f"Training: {len(train_df):,} samples")
print(f"Testing: {len(test_df):,} samples")
print(f"Categories: {len(label_mapping)}")

print(f"\n🎯 Category merges applied:")
print(f"   • Gaming → Entertainment (gaming is entertainment)")
print(f"   • Education → Science & Technology (educational tech)")
print(f"   • Film & Animation → Entertainment")

print(f"\n📂 Files saved:")
print(f"   • train_data.csv")
print(f"   • test_data.csv")
print(f"   • label_mapping.json")

print("\n✅ Ready for training!")

FINAL PREPROCESSING - 8 STRONG CATEGORIES
Original: 40,949

🔀 Merging similar/weak categories...
After merging: 38,558

Merged categories:
category_merged
Comedy                   3391
Entertainment           12970
Howto & Style            4141
Music                    6417
News & Politics          2458
People & Blogs           3048
Science & Technology     3970
Sports                   2163
Name: count, dtype: int64

🔍 Deduplicating...
After dedup: 24,185

⚖️ Sampling per category...
   Comedy: 2112 samples
   Entertainment: 2500 samples
   Howto & Style: 2500 samples
   Music: 2500 samples
   News & Politics: 1816 samples
   People & Blogs: 1985 samples
   Science & Technology: 2500 samples
   Sports: 1536 samples

✅ Final dataset: 17,449 videos

Final distribution:
category_merged
Comedy                  2112
Entertainment           2500
Howto & Style           2500
Music                   2500
News & Politics         1816
People & Blogs          1985
Science & Technology    2500
Sp